# 05 — Bias Analysis in Scientific Document Retrieval

This notebook analyzes potential sources of bias in the NLP Knowledge Discovery Platform.

The project uses scientific documents from arXiv. Even though this is not a social media or demographic dataset, bias can still appear through:

- category imbalance,
- overrepresentation of certain research topics,
- dominance of popular methods such as transformers,
- retrieval bias caused by keyword overlap,
- embedding bias caused by semantic model pretraining,
- knowledge graph bias caused by highly connected concepts.

The goal of this notebook is not to fully solve bias, but to inspect where bias could enter the system and provide evidence for the final project discussion.

## Imports and setup

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.utils.common import (
    display_basic_frame_info,
    load_or_create_processed_documents,
    setup_notebook,
)

CONFIG, PATHS = setup_notebook()

## Load processed documents

In [ ]:
df = load_or_create_processed_documents(CONFIG, PATHS)

display_basic_frame_info(df, "Processed documents")

## Validate available columns

In [ ]:
required_columns = {"doc_id", "title", "abstract", "categories"}

available_columns = set(df.columns)
missing_columns = required_columns - available_columns

print("Available columns:", sorted(available_columns))

if missing_columns:
    print("Missing optional analysis columns:", missing_columns)
else:
    print("All required analysis columns are available.")

## Category parsing helper

In [ ]:
def parse_categories(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if not text:
        return []

    # Handles strings like "['cs.AI', 'cs.CL']"
    if text.startswith("[") and text.endswith("]"):
        try:
            import ast
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed]
        except Exception:
            pass

    # Handles arXiv format: "cs.AI cs.LG"
    return [item.strip() for item in text.split() if item.strip()]

## Category distribution

In [ ]:
category_counter = Counter()

if "categories" in df.columns:
    for categories in df["categories"]:
        category_counter.update(parse_categories(categories))

category_distribution = (
    pd.DataFrame(category_counter.items(), columns=["category", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(category_distribution.head(20))

## Category imbalance visualization

In [ ]:
if category_distribution.empty:
    print("No category information available.")
else:
    top_categories = category_distribution.head(15).sort_values("count")

    plt.figure(figsize=(10, 6))
    plt.barh(top_categories["category"], top_categories["count"])
    plt.title("Top arXiv Categories in the Current Dataset")
    plt.xlabel("Number of Documents")
    plt.ylabel("Category")
    plt.tight_layout()
    plt.show()

## Compute category imbalance metrics

In [ ]:
if category_distribution.empty:
    imbalance_summary = pd.DataFrame()
else:
    total = category_distribution["count"].sum()
    category_distribution["share"] = category_distribution["count"] / total

    top_category_share = category_distribution.iloc[0]["share"]
    top_5_share = category_distribution.head(5)["share"].sum()

    imbalance_summary = pd.DataFrame(
        [
            {
                "metric": "number_of_categories",
                "value": len(category_distribution),
            },
            {
                "metric": "top_category_share",
                "value": top_category_share,
            },
            {
                "metric": "top_5_category_share",
                "value": top_5_share,
            },
        ]
    )

display(imbalance_summary)

## Topic/method keyword bias analysis

In [ ]:
bias_terms = {
    "transformer": ["transformer", "bert", "gpt", "attention"],
    "graph": ["graph", "knowledge graph", "gnn", "graph neural"],
    "retrieval": ["retrieval", "search", "ranking", "recommendation"],
    "vision": ["image", "vision", "segmentation", "object detection"],
    "medical": ["medical", "clinical", "health", "biomedical"],
    "fairness": ["bias", "fairness", "debiasing", "ethics"],
}

text_series = (
    df.get("title", "").fillna("").astype(str)
    + " "
    + df.get("abstract", "").fillna("").astype(str)
).str.lower()

term_rows = []

for group, terms in bias_terms.items():
    mask = pd.Series(False, index=df.index)

    for term in terms:
        mask = mask | text_series.str.contains(term, regex=False)

    term_rows.append(
        {
            "term_group": group,
            "documents": int(mask.sum()),
            "share": float(mask.mean()),
        }
    )

term_bias_df = (
    pd.DataFrame(term_rows)
    .sort_values("documents", ascending=False)
    .reset_index(drop=True)
)

display(term_bias_df)

## Visualize method/topic dominance

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(term_bias_df["term_group"], term_bias_df["documents"])
plt.title("Presence of Selected Topic/Method Terms")
plt.xlabel("Term Group")
plt.ylabel("Number of Documents")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Retrieval result bias analysis

In [ ]:
result_paths = {
    "BM25": PATHS.reports / "tables" / "bm25_results.csv",
    "Semantic": PATHS.reports / "tables" / "semantic_results.csv",
    "KG-enhanced": PATHS.reports / "tables" / "kg_enhanced_results.csv",
}

loaded_results = {}

for method, path in result_paths.items():
    if path.exists():
        loaded_results[method] = pd.read_csv(path, dtype={"doc_id": "string"})
        print(f"Loaded {method} results:", path)
    else:
        print(f"Missing {method} results:", path)

## Analyze repeated documents in retrieval results

In [ ]:
retrieval_bias_rows = []

for method, results in loaded_results.items():
    if "doc_id" not in results.columns:
        continue

    doc_counts = results["doc_id"].astype(str).value_counts()

    retrieval_bias_rows.append(
        {
            "method": method,
            "unique_retrieved_docs": int(doc_counts.shape[0]),
            "total_retrieved_rows": int(len(results)),
            "most_frequent_doc_count": int(doc_counts.iloc[0]) if not doc_counts.empty else 0,
            "top_doc_share": float(doc_counts.iloc[0] / len(results)) if len(results) else 0.0,
        }
    )

retrieval_bias_df = pd.DataFrame(retrieval_bias_rows)

display(retrieval_bias_df)

## Join retrieval results with categories

In [ ]:
doc_metadata = df[["doc_id", "title", "categories"]].copy()
doc_metadata["doc_id"] = doc_metadata["doc_id"].astype(str)

retrieval_category_rows = []

for method, results in loaded_results.items():
    merged = results.copy()
    merged["doc_id"] = merged["doc_id"].astype(str)

    merged = merged.merge(
        doc_metadata,
        on="doc_id",
        how="left",
        suffixes=("", "_document"),
    )

    for _, row in merged.iterrows():
        for category in parse_categories(row.get("categories", "")):
            retrieval_category_rows.append(
                {
                    "method": method,
                    "query_id": row.get("query_id"),
                    "doc_id": row.get("doc_id"),
                    "category": category,
                }
            )

retrieval_categories_df = pd.DataFrame(retrieval_category_rows)

display(retrieval_categories_df.head())

## Retrieved category distribution

In [ ]:
if retrieval_categories_df.empty:
    print("No retrieval-category data available.")
else:
    retrieved_category_distribution = (
        retrieval_categories_df
        .groupby(["method", "category"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values(["method", "count"], ascending=[True, False])
    )

    display(retrieved_category_distribution.head(30))

## Interpretation

The bias analysis suggests that this project can contain several forms of bias even without demographic data.

Potential bias sources:

1. **Dataset bias**  
   If a few arXiv categories dominate the dataset, then topic modeling, retrieval, and graph construction will mainly represent these categories.

2. **Method popularity bias**  
   Popular terms such as "transformer", "graph", or "retrieval" may dominate the extracted keywords and graph concepts.

3. **Retrieval bias**  
   A small number of highly ranked documents may appear repeatedly across many queries.

4. **Graph centrality bias**  
   Highly connected concepts can dominate KG-enhanced retrieval, even when they are not always the most relevant.

5. **Embedding model bias**  
   Sentence Transformer embeddings are pretrained on large external corpora and may encode domain or language-specific biases.

For this project, the most relevant mitigation strategy is not full debiasing, but transparent analysis:
- report category distributions,
- inspect repeated retrieved documents,
- compare BM25, semantic retrieval, and KG-enhanced retrieval,
- discuss limitations in the final report.

## Save bias analysis outputs

In [ ]:
output_dir = PATHS.reports / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

if not category_distribution.empty:
    category_distribution.to_csv(output_dir / "bias_category_distribution.csv", index=False)

if not imbalance_summary.empty:
    imbalance_summary.to_csv(output_dir / "bias_imbalance_summary.csv", index=False)

term_bias_df.to_csv(output_dir / "bias_term_presence.csv", index=False)

if not retrieval_bias_df.empty:
    retrieval_bias_df.to_csv(output_dir / "bias_retrieval_repetition.csv", index=False)

if "retrieved_category_distribution" in globals():
    retrieved_category_distribution.to_csv(
        output_dir / "bias_retrieved_category_distribution.csv",
        index=False,
    )

print("Bias analysis outputs saved to:", output_dir)